# 04-2. 확정 규칙 기반 데이터 정제

같은 4어절이 3회 이상 반복되는 summary 행을 제외하고, document에서 URL·명확한 저작권/방송 안내 문구·깨진 대체문자만 제거한다. 기자명과 언론사명은 보존한다.

In [1]:
import os, re, sys
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi

PROJECT=Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert Path(sys.prefix).resolve()==(PROJECT/".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
os.environ["HF_HOME"]=str(PROJECT/".cache"/"huggingface")
DATASET_ID="daekeun-ml/naver-news-summarization-ko"
OUTPUT=PROJECT/"processed_data"/"04-2_cleaned"
RESULTS=PROJECT/"review_data"/"04-2_cleaning"
OUTPUT.mkdir(parents=True,exist_ok=True); RESULTS.mkdir(parents=True,exist_ok=True)
word_pattern=re.compile(r"[가-힣A-Za-z0-9]+(?:[.,]\d+)?")


In [2]:
def max_4gram(text):
    tokens=word_pattern.findall(str(text).lower())
    counts=Counter(tuple(tokens[i:i+4]) for i in range(len(tokens)-3))
    return max(counts.values(),default=0)
URL=re.compile(r"(?i)\b(?:https?://|www\.)[^\s<>()\[\]{}]+")
BOILERPLATES=[
 re.compile(r"국내에서만\s*사용가능\.?\s*재배포\s*금지\.?\s*DB\s*금지\.?\s*For\s*Use\s*Only\s*in\s*the\s*Republic\s*of\s*Korea\.?\s*(?:No\s*)?Redistribution\s*Prohibited",re.I),
 re.compile(r"무단\s*(?:전재|복제|배포)\s*(?:및\s*)?재배포\s*금지\.?",re.I),
 re.compile(r"재배포\s*금지\.?",re.I),
 re.compile(r"인터뷰\s*자료의\s*저작권은\s*[^.]{0,120}(?:있습니다|있다)\.?",re.I),
 re.compile(r"(?:전문\s*게재나?\s*)?인터뷰\s*인용\s*보도\s*시\s*[^.]{0,180}(?:바랍니다|바란다)\.?",re.I),
 re.compile(r"인용\s*보도\s*시\s*출처를\s*[^.]{0,120}(?:바랍니다|바란다)\.?",re.I),
 re.compile(r"아래\s*텍스트는\s*실제\s*방송\s*내용과\s*차이가\s*있을\s*수\s*있으니\s*[^.]{0,180}(?:바랍니다|바란다)\.?",re.I)]
def clean_document(text):
    original=str(text); cleaned,url_n=URL.subn("",original); boiler_n=0
    for pattern in BOILERPLATES:
        cleaned,n=pattern.subn("",cleaned); boiler_n+=n
    replacement_n=cleaned.count("�"); cleaned=cleaned.replace("�"," ")
    cleaned=re.sub(r"[ \t]+"," ",cleaned); cleaned=re.sub(r" *\n *","\n",cleaned).strip()
    return cleaned,url_n,boiler_n,replacement_n

revision=HfApi().dataset_info(DATASET_ID).sha; dataset=load_dataset(DATASET_ID,revision=revision)
frames=[]
for split in ["train","validation","test"]:
    frame=dataset[split].to_pandas().copy(); frame.insert(0,"source_index",np.arange(len(frame)))
    frame.insert(0,"split",split); frame.insert(0,"row_id",split+":"+frame.source_index.astype(str))
    frame["max_same_4gram_occurrences"]=frame.summary.map(max_4gram); frame["repeat_4gram"]=frame.max_same_4gram_occurrences.ge(3)
    frames.append(frame)
raw=pd.concat(frames,ignore_index=True); excluded=raw[raw.repeat_4gram].copy(); clean=raw[~raw.repeat_4gram].copy()
processed=clean.document.map(clean_document); clean.insert(clean.columns.get_loc("document")+1,"document_original",clean.document)
clean["document"]=processed.map(lambda x:x[0]); clean["url_removed"]=processed.map(lambda x:x[1])
clean["boilerplate_removed"]=processed.map(lambda x:x[2]); clean["replacement_char_removed"]=processed.map(lambda x:x[3])
clean["document_changed"]=clean.document.ne(clean.document_original)
assert clean.document.str.strip().ne("").all()


In [3]:
for split,frame in clean.groupby("split",sort=False): frame.to_csv(OUTPUT/f"{split}.csv",index=False,encoding="utf-8-sig")
excluded[["row_id","split","title","link","document","summary","max_same_4gram_occurrences"]].to_csv(RESULTS/"excluded_repeat_4gram_rows.csv",index=False,encoding="utf-8-sig")
changed=clean[clean.document_changed][["row_id","split","title","document_original","document","url_removed","boilerplate_removed","replacement_char_removed"]].copy()
changed.to_csv(RESULTS/"changed_document_comparison.csv",index=False,encoding="utf-8-sig")
summary=[]
for split in ["train","validation","test"]:
    before=raw[raw.split.eq(split)]; after=clean[clean.split.eq(split)]
    summary.append({"split":split,"original_rows":len(before),"repeat_excluded":int(before.repeat_4gram.sum()),"clean_rows":len(after),"documents_changed":int(after.document_changed.sum()),"urls_removed":int(after.url_removed.sum()),"boilerplates_removed":int(after.boilerplate_removed.sum()),"replacement_chars_removed":int(after.replacement_char_removed.sum())})
summary=pd.DataFrame(summary); summary.to_csv(RESULTS/"cleaning_summary.csv",index=False,encoding="utf-8-sig")
pd.DataFrame([{"dataset_id":DATASET_ID,"dataset_revision":revision,"output_data":str(OUTPUT),"policy":"drop repeat_4gram>=3; remove URL/explicit boilerplate/U+FFFD; preserve press and reporter names"}]).to_csv(RESULTS/"run_info.csv",index=False,encoding="utf-8-sig")
display(summary)

,split,original_rows,repeat_excluded,clean_rows,documents_changed,urls_removed,boilerplates_removed,replacement_chars_removed
0,train,22194,1121,21073,104,73,52,7
1,validation,2466,121,2345,9,7,8,0
2,test,2740,134,2606,16,20,2,0
